# Introduction

This project aims to improve the efficiency of robust simulation-based inference, proposed by Huang et al. in 2023. Specifically, it does so by improving on the maximum-mean-discrepancy (MMD) metric that the previous paper uses to regularize the model summarizers. We investigate the implementation of sample-efficient MMD and quasi-Monte Carlo (QMC) methods to improve the computation of this metric. 

In [1]:
import sys
sys.path.append('../')
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pickle
import matplotlib
import matplotlib.pyplot as plt
# import seaborn as sns
from itertools import permutations
import random
import time
import os

from utils.metrics import RMSE
import utils.metrics as metrics
from simulators.ricker import ricker

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

import warnings
warnings.filterwarnings('ignore')

# Below: new imports
from utils.timer import Timer

print(device)

c:\Users\azhao\.conda\envs\efficient-sbi\lib\site-packages\arviz\data\base.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
<frozen importlib._bootstrap>:228: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 88 from PyObject


cuda:0


## Experiment 1: Sample-efficient MMD on Ricker ABC

For SBI, the choice of summary statistics is crucial to specify the proper distribution for data. For instance, a Gaussian distribution is poorly specified to bimodal data, but with only mean and variance, there may be a distribution of each type where those summary statistics match. Thus, robust SBI trains a model summarizer to minimize the MMD. 

Computing MMD as a loss function is computationally expensive. We aim to optimize this by using sample-efficient MMD to improve the overall runtime. 

For this experiment, we use approximate Bayesian computation (ABC) as the SBI method of interest. We evaluate three approaches for ABC: the normal approach, the robust approach, and the efficient approach.

### Load Data

The data is generated with `generate_data.py` with 1000 simulations and 100 realizations. 

In [ ]:
# Number of samples m
m = 1000

# Load x
x = torch.tensor(np.load(f"../data/ricker_x_{m}.npy")).reshape(m, 100, 100).to(device)

# load parameters theta
theta = np.load(f"../data/ricker_theta_{m}.npy")

dataloader = DataLoader(x[:1000], batch_size=200, shuffle=True)

### Define regression-based ABC model

Using observations, input parameters, a set of summary statistics, and 

In [ ]:
def regression_ABC(s_obs, param, sumStats, p):
    def mad(data):
        return np.mean(np.abs(data - np.mean(data, axis=0)), axis=0)

    if param.shape[0] < param.shape[1]:
        param = np.transpose(param)

    if sumStats.shape[0] < sumStats.shape[1]:
        sumStats = np.transpose(sumStats)
    
    M = len(param)
    M_epsilon = int(M*p)
    sumStats = sumStats
    s_obs = s_obs

    norm_factor = mad(sumStats)

    norm_sumStats = sumStats / norm_factor
    norm_s_obs = s_obs / norm_factor

    distance = np.linalg.norm(norm_sumStats - norm_s_obs, axis = 1)
    max_accepted_distance = np.sort(distance)[M_epsilon - 1]

    posterior_samples = param[distance <= max_accepted_distance, :]
    norm_sumStats_star = norm_sumStats[distance <= max_accepted_distance, :]

    weights = 1 - (distance[distance <= max_accepted_distance] / max_accepted_distance)**2
    W = np.diag(weights)

    s_obs_norm = np.tile(norm_s_obs, (M_epsilon,1))
    X = np.column_stack((np.ones(shape = (M_epsilon,1)), norm_sumStats_star - s_obs_norm))

    A = np.matmul(X.T, W)

    solution = np.linalg.solve(np.matmul(A, X), np.matmul(A, posterior_samples))

    beta = solution[1:,:]

    posterior_samples_adjusted = posterior_samples - np.matmul((norm_sumStats_star - s_obs_norm), beta)

    return posterior_samples_adjusted
    

### Define summary network

In [ ]:
class RickerSummary(nn.Module):
    def __init__(self, input_size, hidden_dim):
        super(RickerSummary, self).__init__()

        self.hidden_dim = hidden_dim
        self.input_size = input_size
        
        self.encoder = nn.Sequential(nn.Conv1d(self.input_size, 4, 3, 4),
                                     nn.Conv1d(4, 4, 3, 4),
                                     nn.Conv1d(4, 4, 3, 4),
                                     )
        
        self.decoder = nn.Sequential(nn.ConvTranspose1d(4, 4, 3, 4),
                                     nn.ConvTranspose1d(4, 4, 3, 4),
                                     nn.ConvTranspose1d(4, self.input_size, 3, 4),
                                     nn.Upsample(100)
                                     )

    def forward(self, Y):
        embeddings = self.encoder(Y.reshape(-1, 1, 100))
        output = self.decoder(embeddings.reshape(-1, 4, 1)).reshape(-1, 100, 100)
        return output
    
    def forward_encoder(self, Y):
        embeddings = self.encoder(Y.reshape(-1, 1, 100)).reshape(-1, 100, 4)
        return embeddings

### Define solver

In order to evaluate each method, we need to solve the SBI and produce a posterior distribution. We can define a modular solver that uses different methods to produce our output. We can use the aforementioned `RickerSummary()` to define the solver.

In [7]:
def solve(x, beta, obs_cont, mmd_metric=None, root=".", max_epochs=10, patience=3, lr=0.01, verbose=False):
    """
    General solver function for Ricker summary statistics. 
    Use mmd_metric to differentiate between robust and efficient-robust implementations.
    Set beta to 0 for a non-regularized "normal" solve.
    Timer laps once per epoch.
    """
    summary_net = RickerSummary(1, 4).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(summary_net.parameters(), lr=lr)

    index_list = [int(i) for i in range(len(x))]

    name = f"{mmd_metric.__name__ if mmd_metric is not None else 'normal'}_lambda={beta}_obscont={obs_cont}"

    timer = Timer(name, root)
    timer.start()
    previous_loss = np.inf
    epochs_without_descent = 0
    for epoch in range(max_epochs):
        running_loss = 0.0

        for data in dataloader:
            X = data
            optimizer.zero_grad()

            Y = summary_net(X)

            random.shuffle(index_list)
            context_embeddings = torch.mean(summary_net.forward_encoder(x[index_list[:200]]), dim=1)
            obs_embeddings = torch.mean(summary_net.forward_encoder(obs_cont), dim=1)

            l_scale = metrics.median_heuristic_combined(context_embeddings, obs_embeddings)

            ae_loss = criterion(Y, X) / 10000
            summary_loss = metrics.MMD_weighted(context_embeddings, 
                                                obs_embeddings, 
                                                lengthscale=l_scale)

            loss = ae_loss + beta*summary_loss

            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        if (running_loss < previous_loss):
            epochs_without_descent += 1
        else:
            previous_loss = running_loss
        
        if (epochs_without_descent > patience):
            break

        timer.lap(verbose=verbose)
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader)}")
    timer.stop()
    return summary_net